### Experiment 4 Binary Classification on Spambase using Logistic Regression and SVM
#### Hemnath D - 3122247001022 (MTech Integrated CSE)

Aim: To perform binary classification on the Spambase dataset using Logistic Regression and SVM, including hyperparameter tuning and a comparison of both models

In [ ]:
import sys, os, time, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.feature_selection import f_classif
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

%matplotlib inline


##### Helper functions for EDA
These are the same EDA utilities used in Experiment 1 (missing value checks, correlation, normality tests, outlier detection, variance/feature-importance ranking, pairwise plots), kept here directly so this notebook runs on its own without needing the Experiment 1 folder.

In [ ]:
import os
from scipy import stats
from sklearn.feature_selection import mutual_info_classif, VarianceThreshold
from sklearn.covariance import EllipticEnvelope
from sklearn.neighbors import LocalOutlierFactor

os.makedirs("images", exist_ok=True)


def check_missing_values(df, name):
    missing = df.isnull().sum()
    missing_rate = df.isnull().mean().mean() * 100
    n_missing_cols = int((missing > 0).sum())

    plt.figure(figsize=(10, 3))
    sns.heatmap(df.isnull(), cbar=True, cmap="viridis", yticklabels=False)
    plt.title(f"Missing Value Matrix - {name}")
    plt.tight_layout()
    plt.savefig(f"images/expt1-{name}-missing.png", dpi=150)
    plt.show()

    print(f"Overall missing rate is {missing_rate:.2f}% across all cells")
    print(f"{n_missing_cols} column(s) contain missing values")


def plot_correlation(df, name):
    corr = df.corr(numeric_only=True)
    plt.figure(figsize=(16, 14))
    sns.heatmap(corr, cmap="coolwarm", center=0, square=True, cbar=True,
                xticklabels=False, yticklabels=False)
    plt.title(f"Correlation Heatmap - {name}")
    plt.tight_layout()
    plt.savefig(f"images/expt1-{name}-correlation.png", dpi=150)
    plt.show()

    corr_abs = corr.copy()
    for c in corr_abs.columns:
        corr_abs.loc[c, c] = 0
    max_pair = corr_abs.abs().stack().idxmax()
    max_val = corr.loc[max_pair]
    print(f"Strongest pairwise correlation is between '{max_pair[0]}' and "
          f"'{max_pair[1]}' (r = {max_val:.2f})")
    return corr


def pick_key_columns(df, target_col, max_cols=10):
    feature_cols = [c for c in df.columns if c != target_col]
    skewness = df[feature_cols].skew().abs().sort_values(ascending=False)
    return list(skewness.index[:max_cols])


def check_normality(df, cols, name):
    for col in cols:
        fig, axes = plt.subplots(1, 4, figsize=(14, 2.8))
        sns.histplot(df[col], kde=True, ax=axes[0], color="steelblue")
        axes[0].set_title("Histogram")
        sns.boxplot(x=df[col], ax=axes[1], color="lightcoral")
        axes[1].set_title("Boxplot")
        stats.probplot(df[col], dist="norm", plot=axes[2])
        axes[2].set_title("Q-Q Plot")
        sns.kdeplot(df[col], ax=axes[3], fill=True, color="seagreen")
        axes[3].set_title("Density")
        fig.suptitle(f"Normality Check - {col}")
        plt.tight_layout()
        safe_col = col.replace("/", "-")
        plt.savefig(f"images/expt1-{name}-{safe_col}-normality.png", dpi=150)
        plt.show()

        stat, p_value = stats.shapiro(df[col].sample(min(len(df), 5000), random_state=42))
        verdict = "deviates from normality" if p_value < 0.05 else "looks approximately normal"
        print(f"Shapiro-Wilk test on '{col}': p = {p_value:.4f}, so the distribution "
              f"{verdict} (alpha = 0.05)")


def find_outliers(df, cols, name):
    for col in cols:
        q1, q3 = df[col].quantile(0.25), df[col].quantile(0.75)
        iqr = q3 - q1
        lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
        n_outliers = int(((df[col] < lower) | (df[col] > upper)).sum())

        fig, axes = plt.subplots(1, 2, figsize=(9, 2.8))
        sns.boxplot(x=df[col], ax=axes[0], color="orange")
        axes[0].axvline(lower, color="red", linestyle="--", linewidth=1)
        axes[0].axvline(upper, color="red", linestyle="--", linewidth=1)
        axes[0].set_title("Boxplot with IQR bounds")
        sns.histplot(df[col], ax=axes[1], color="slateblue", kde=False)
        axes[1].axvline(lower, color="red", linestyle="--", linewidth=1)
        axes[1].axvline(upper, color="red", linestyle="--", linewidth=1)
        axes[1].set_title("Distribution with bounds")
        fig.suptitle(f"Outlier Check - {col}")
        plt.tight_layout()
        safe_col = col.replace("/", "-")
        plt.savefig(f"images/expt1-{name}-{safe_col}-outliers.png", dpi=150)
        plt.show()

        print(f"{n_outliers} of {len(df)} values in '{col}' are outliers "
              f"(outside [{lower:.2f}, {upper:.2f}]).")


def find_multivariate_outliers(df, cols, name):
    X = df[cols].values

    ee = EllipticEnvelope(contamination=0.025, random_state=42)
    mahal_flags = ee.fit_predict(X) == -1
    n_mahal = int(mahal_flags.sum())

    lof = LocalOutlierFactor(n_neighbors=20)
    lof_flags = lof.fit_predict(X) == -1
    n_lof = int(lof_flags.sum())

    lof_scores = -lof.negative_outlier_factor_
    plt.figure(figsize=(9, 2.8))
    plt.subplot(1, 2, 1)
    plt.scatter(range(len(X)), lof_scores, s=5, c=["red" if f else "grey" for f in lof_flags])
    plt.title("LOF outlier scores")
    plt.xlabel("Sample index")
    plt.ylabel("LOF score")
    plt.subplot(1, 2, 2)
    plt.bar(["Mahalanobis", "LOF"], [n_mahal, n_lof], color=["indianred", "cornflowerblue"])
    plt.title("Flagged point counts")
    plt.tight_layout()
    plt.savefig(f"images/expt1-{name}-multivariate-outliers.png", dpi=150)
    plt.show()

    print(f"Mahalanobis distance flags {n_mahal} points as outliers across correlated "
          f"features; LOF flags {n_lof} points as locally isolated")


def check_variance(df, target_col, name, threshold=0.1):
    feature_cols = [c for c in df.columns if c != target_col]
    variances = df[feature_cols].var().sort_values()
    n_below = int((variances < threshold).sum())

    plt.figure(figsize=(14, 4))
    colors = ["crimson" if v < threshold else "steelblue" for v in variances]
    plt.bar(range(len(variances)), variances.values, color=colors)
    plt.axhline(threshold, color="black", linestyle="--", linewidth=1)
    plt.title(f"Feature Variance - {name}")
    plt.xlabel("Feature index (sorted)")
    plt.ylabel("Variance")
    plt.tight_layout()
    plt.savefig(f"images/expt1-{name}-variance.png", dpi=150)
    plt.show()

    print(f"{n_below} of {len(feature_cols)} features fall below the variance threshold of {threshold}")


def rank_features(df, target_col, score_func, name):
    feature_cols = [c for c in df.columns if c != target_col]
    X = df[feature_cols].values
    y = df[target_col].values
    scores, _ = score_func(X, y)
    ranking = pd.Series(scores, index=feature_cols).sort_values(ascending=True)

    plt.figure(figsize=(10, 8))
    ranking.plot(kind="barh", color="teal")
    plt.title(f"ANOVA F-Score Feature Ranking - {name}")
    plt.xlabel("F-score")
    plt.tight_layout()
    plt.savefig(f"images/expt1-{name}-importance.png", dpi=150)
    plt.show()

    top_feature = ranking.idxmax()
    print(f"'{top_feature}' has the strongest relationship with the target "
          f"(score: {ranking.max():.2f})")
    return ranking


def rank_mutual_info(df, target_col, name, task="classification"):
    feature_cols = [c for c in df.columns if c != target_col]
    X = df[feature_cols].values
    y = df[target_col].values
    mi = mutual_info_classif(X, y, random_state=42)
    ranking = pd.Series(mi, index=feature_cols).sort_values(ascending=True)

    plt.figure(figsize=(10, 8))
    ranking.plot(kind="barh", color="darkorange")
    plt.title(f"Mutual Information Ranking - {name}")
    plt.xlabel("Mutual information")
    plt.tight_layout()
    plt.savefig(f"images/expt1-{name}-mutual-info.png", dpi=150)
    plt.show()

    top_feature = ranking.idxmax()
    print(f"'{top_feature}' shares the most information with the target ({ranking.max():.3f})")
    return ranking


def plot_pairwise(df, cols, target_col, name):
    plot_cols = cols + [target_col]
    g = sns.pairplot(df[plot_cols], hue=target_col, palette="deep", diag_kind="hist",
                      plot_kws={"s": 8, "alpha": 0.5})
    g.fig.suptitle(f"Pairwise Feature Relationships - {name}", y=1.02)
    g.savefig(f"images/expt1-{name}-pairwise.png", dpi=120)
    plt.show()


In [ ]:
def load_dataset(path):
    return pd.read_csv(path)

In [ ]:
def standardize_features(df, target_col):
    # scales every column except the target using a StandardScaler fit on the whole frame
    df = df.copy()
    feature_cols = [c for c in df.columns if c != target_col]
    scaler = StandardScaler()
    df[feature_cols] = scaler.fit_transform(df[feature_cols])
    return df

In [ ]:
def plot_class_balance(df, target_col, dataset):
    counts = df[target_col].value_counts().sort_index()

    plt.figure(figsize=(4, 3.2))
    sns.barplot(x=counts.index.astype(str), y=counts.values, hue=counts.index.astype(str),
                palette="deep", legend=False)
    plt.title(f"Class Balance ({dataset})")
    plt.xlabel(target_col)
    plt.ylabel("Count")
    plt.tight_layout()
    plt.savefig(os.path.join("images", f"expt4-{dataset}-class-balance.png"), dpi=600)
    plt.show()

    return counts.rename("count").reset_index().rename(columns={"index": target_col})

In [ ]:
def split_dataset(df, target_col, test_size=0.2, random_state=42):
    X = df.drop(columns=[target_col])
    y = df[target_col]
    return train_test_split(X, y, test_size=test_size, stratify=y, random_state=random_state)

In [ ]:
def fit_and_time(model, X_train, y_train):
    start = time.time()
    model.fit(X_train, y_train)
    elapsed = time.time() - start
    return model, elapsed

In [ ]:
def evaluate_model(model, X_test, y_test):
    preds = model.predict(X_test)
    return {
        "Accuracy": accuracy_score(y_test, preds),
        "Precision": precision_score(y_test, preds),
        "Recall": recall_score(y_test, preds),
        "F1 Score": f1_score(y_test, preds),
    }

In [ ]:
def train_and_evaluate(model, X_train, y_train, X_test, y_test, model_name):
    fitted_model, elapsed = fit_and_time(model, X_train, y_train)
    metrics = evaluate_model(fitted_model, X_test, y_test)
    row = {"Model": model_name, **metrics, "Training Time (s)": elapsed}
    return pd.DataFrame([row]), fitted_model

In [ ]:
def tune_hyperparameters(estimator, param_grid, X_train, y_train, cv=5, scoring="accuracy",
                          search_method="grid", n_iter=30, random_state=42):
    if search_method == "grid":
        search = GridSearchCV(estimator, param_grid, cv=cv, scoring=scoring, n_jobs=-1)
    else:
        search = RandomizedSearchCV(estimator, param_grid, n_iter=n_iter, cv=cv, scoring=scoring,
                                     n_jobs=-1, random_state=random_state)
    search.fit(X_train, y_train)
    return search

In [ ]:
def build_tuning_summary(search, model_name, search_method):
    row = {
        "Model": model_name,
        "Search Method": search_method,
        "Best Parameters": str(search.best_params_),
        "Best CV Accuracy": search.best_score_,
    }
    return pd.DataFrame([row])

In [ ]:
def combine_tables(*dfs):
    return pd.concat(list(dfs), ignore_index=True)

In [ ]:
def run_svm_kernels(X_train, y_train, X_test, y_test, kernels, **svc_kwargs):
    rows = []
    fitted_models = {}
    for kernel in kernels:
        model = SVC(kernel=kernel, random_state=42, **svc_kwargs)
        metrics_df, fitted_model = train_and_evaluate(model, X_train, y_train, X_test, y_test, kernel)
        fitted_models[kernel] = fitted_model
        rows.append(metrics_df)
    kernel_df = combine_tables(*rows).rename(columns={"Model": "Kernel"})
    kernel_df = kernel_df[["Kernel", "Accuracy", "F1 Score", "Training Time (s)"]]
    return kernel_df, fitted_models

In [ ]:
def plot_kernel_comparison(kernel_df):
    fig, axes = plt.subplots(1, 2, figsize=(9, 3.2))
    sns.barplot(data=kernel_df, x="Kernel", y="Accuracy", hue="Kernel", palette="crest", legend=False, ax=axes[0])
    axes[0].set_title("Accuracy by Kernel")
    sns.barplot(data=kernel_df, x="Kernel", y="Training Time (s)", hue="Kernel", palette="flare", legend=False, ax=axes[1])
    axes[1].set_title("Training Time by Kernel")
    plt.tight_layout()
    plt.savefig(os.path.join("images", "expt4-spambase-kernel-comparison.png"), dpi=600)
    plt.show()
    return kernel_df

In [ ]:
def plot_model_comparison(results_df, metrics):
    plot_df = results_df.melt(id_vars="Model", value_vars=metrics, var_name="Metric", value_name="Score")

    plt.figure(figsize=(7, 3.5))
    sns.barplot(data=plot_df, x="Metric", y="Score", hue="Model", palette="deep")
    plt.title("Logistic Regression vs SVM")
    plt.ylim(0, 1.05)
    plt.tight_layout()
    plt.savefig(os.path.join("images", "expt4-spambase-model-comparison.png"), dpi=600)
    plt.show()
    return results_df

In [ ]:
def get_cv_scores(model, X, y, cv=5, scoring="accuracy"):
    return cross_val_score(model, X, y, cv=cv, scoring=scoring)

In [ ]:
def build_cv_table(scores_dict, cv=5):
    table = pd.DataFrame(scores_dict, index=[f"Fold {i+1}" for i in range(cv)])
    table.loc["Average"] = table.mean()
    return table.reset_index().rename(columns={"index": "Fold"})

In [ ]:
def build_comparative_table(lr_row, svm_row):
    lr_acc = lr_row["Accuracy"].iloc[0]
    svm_acc = svm_row["Accuracy"].iloc[0]
    data = {
        "Criterion": ["Accuracy", "Model Complexity", "Training Time", "Interpretability"],
        "Logistic Regression": [f"{lr_acc:.4f}", "Low", "Low", "High"],
        "SVM": [f"{svm_acc:.4f}", "High", "High", "Low"],
    }
    return pd.DataFrame(data)

##### Loading the dataset
Spambase has 4601 emails, each with 57 numeric features describing word/character frequencies and capital-letter run lengths, plus a class label (1 = spam, 0 = ham)

In [ ]:
df = load_dataset("spambase.csv")
print(df.shape)
df.head()

##### Preprocess the data
Checking for missing values first, then standardizing all the feature columns

In [ ]:
check_missing_values(df, "spambase")

In [ ]:
df_scaled = standardize_features(df, "class")
df_scaled.head()

##### Exploratory data analysis

In [ ]:
plot_class_balance(df_scaled, "class", "spambase")

In [ ]:
plot_correlation(df_scaled, "spambase")

In [ ]:
key_cols = pick_key_columns(df_scaled, target_col="class", max_cols=10)
key_cols

In [ ]:
check_normality(df_scaled, key_cols, "spambase")

In [ ]:
find_outliers(df_scaled, key_cols, "spambase")

In [ ]:
find_multivariate_outliers(df_scaled, key_cols, "spambase")

In [ ]:
check_variance(df_scaled, "class", "spambase", threshold=0.1)

In [ ]:
rank_features(df_scaled, "class", f_classif, "spambase")

In [ ]:
rank_mutual_info(df_scaled, "class", "spambase", task="classification")

In [ ]:
plot_pairwise(df_scaled, key_cols, "class", "spambase")

##### Spliting the dataset into training and testing sets
An 80/20 stratified split, so the spam/ham ratio stays consistent between train and test.

In [ ]:
X_train, X_test, y_train, y_test = split_dataset(df_scaled, "class", test_size=0.2, random_state=42)
X_train.shape, X_test.shape

##### Baseline Logistic Regression

In [ ]:
baseline_lr_metrics, baseline_lr_model = train_and_evaluate(
    LogisticRegression(max_iter=1000, random_state=42),
    X_train, y_train, X_test, y_test, "Logistic Regression (baseline)"
)
baseline_lr_metrics

##### Tuning Logistic Regression hyperparameters
Grid search over the regularization type

In [ ]:
lr_param_grid = {
    "penalty": ["l1", "l2"],
    "C": [0.01, 0.1, 1, 10, 100],
    "solver": ["liblinear", "saga"],
}
lr_search = tune_hyperparameters(
    LogisticRegression(max_iter=5000, random_state=42), lr_param_grid,
    X_train, y_train, cv=5, search_method="grid"
)
lr_tuning_summary = build_tuning_summary(lr_search, "Logistic Regression", "Grid")
lr_tuning_summary

In [ ]:
best_lr_model = lr_search.best_estimator_
lr_metrics, _ = train_and_evaluate(best_lr_model, X_train, y_train, X_test, y_test, "Logistic Regression (tuned)")
lr_performance_table = lr_metrics.drop(columns="Model").T.reset_index()
lr_performance_table.columns = ["Metric", "Value"]
lr_performance_table

##### Train SVM with different kernels

In [ ]:
svm_kernel_df, svm_kernel_models = run_svm_kernels(X_train, y_train, X_test, y_test,
                                                     kernels=["linear", "poly", "rbf", "sigmoid"])
svm_kernel_df

In [ ]:
plot_kernel_comparison(svm_kernel_df)

##### Tune SVM hyperparameters
Grid search across C, gamma, and (for the polynomial kernel) degree, searched separately per kernel

In [ ]:
svm_param_grid = [
    {"kernel": ["linear"], "C": [0.1, 1, 10, 100]},
    {"kernel": ["poly"], "C": [0.1, 1, 10, 100], "gamma": ["scale", "auto"], "degree": [2, 3, 4]},
    {"kernel": ["rbf"], "C": [0.1, 1, 10, 100], "gamma": ["scale", "auto"]},
    {"kernel": ["sigmoid"], "C": [0.1, 1, 10, 100], "gamma": ["scale", "auto"]},
]
svm_search = tune_hyperparameters(SVC(random_state=42), svm_param_grid, X_train, y_train, cv=5, search_method="grid")
svm_tuning_summary = build_tuning_summary(svm_search, "SVM", "Grid")
tuning_results = combine_tables(lr_tuning_summary, svm_tuning_summary)
tuning_results

##### Evaluate models using standard metrics

In [ ]:
best_svm_model = svm_search.best_estimator_
svm_metrics, _ = train_and_evaluate(best_svm_model, X_train, y_train, X_test, y_test, "SVM (tuned)")
final_comparison = combine_tables(lr_metrics, svm_metrics)
final_comparison

In [ ]:
plot_model_comparison(final_comparison, metrics=["Accuracy", "Precision", "Recall", "F1 Score"])

##### Perform 5-fold cross-validation
Running the two tuned models through 5-fold CV on the training data

In [ ]:
cv_scores_dict = {
    "Logistic Regression": get_cv_scores(best_lr_model, X_train, y_train, cv=5),
    "SVM": get_cv_scores(best_svm_model, X_train, y_train, cv=5),
}
cv_table = build_cv_table(cv_scores_dict, cv=5)
cv_table

##### Comparative analysis


In [ ]:
comparative_table = build_comparative_table(lr_metrics, svm_metrics)
comparative_table

##### Comparison
- Both tuned models are similar 
- Logistic regression is very slightly ahead on the held out test set (92.9% vs 92.7% accuracy for the SVM)
- But SVM actually had the higher average cross-validation accuracy (93.4% vs 92.4%)
- On balance the RBF-kernel SVM looks better for spam detection, since it generalizes a bit better across folds and trains faster

##### Regularization
- The grid search actually picked L1 (Lasso) regularization with C=1 and the solver over the default L2/lbfgs setup, which means some coefficients got shrunk to exactly zero
- This is useful on a bag-of-word-frequency feature set where a lot of the 57 columns are only weakly informative
- It converges much more slowly than liblinear or lbfgs on this data, which is why the tuned logistic regression took about 8.6 seconds to fit versus roughly 0.04 seconds for the untuned baseline

##### Kernel
- Kernel choice made a bigger difference for the SVM than tuning did
- The linear and RBF kernels were both strong out of the box (93.1% and 92.7% accuracy), the sigmoid kernel trailed a bit (88.4%), and the polynomial kernel was clearly the weak link at only about 78% 
- RBF is the best in the hyperparameter search

##### Bias Variance Trade-off
- Logistic regression is the higher-bias, lower-variance model
- The tuned SVM can bend its decision boundary around the data more, so it trades some of that bias for a bit more variance, which is part of why its fold-to-fold accuracy moves around a little more in the cross-validation table. 